# ROI Budgeting Colab Runner

This notebook runs the full isolated ROI-budgeting experiment flow on Colab:

- prepare the repo and Python environment
- point the run at one test clip
- generate baseline `roi_detections.json` and `frame_drop.json`
- run `v1`, `v2`, and `v3`
- generate real GPU AMT probes for `v3`
- save manifests under `research/roi_budgeting/results/`
- optionally sync those results to Google Drive


## Before You Run

The next cell is preconfigured for a repo stored in Drive at `MyDrive/Colab Notebooks/neural-edge-video-compression`, and the reuse logic will prefer that Drive copy automatically when it exists.

The default test clip path is:

`/content/drive/MyDrive/Colab Notebooks/neural-edge-video-compression/video.mp4`

If your video lives somewhere else in Drive, change only `VIDEO_PATH`. If you want to clone a fresh copy into `/content` instead, set `REPO_URL` back to the GitHub URL and change `REPO_DIR`.


In [ ]:
from pathlib import Path

MOUNT_DRIVE = True
REPO_URL = ""
REPO_DIR = Path("/content/drive/MyDrive/Colab Notebooks/neural-edge-video-compression")
DEFAULT_VIDEO = REPO_DIR / "video.mp4"
VIDEO_PATH = str(DEFAULT_VIDEO)
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/neural-edge-video-compression/research/roi_budgeting/results"
ROI_TARGET_KBPS = 150.0
RUN_V1 = True
RUN_V2 = True
RUN_V3 = True

print({
    "repo_dir": str(REPO_DIR),
    "video_path": VIDEO_PATH,
    "drive_results_dir": DRIVE_RESULTS_DIR,
    "roi_target_kbps": ROI_TARGET_KBPS,
})


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Skipping Drive mount.")


In [ ]:
import os
import subprocess
from pathlib import Path

preferred_drive_repo = Path("/content/drive/MyDrive/Colab Notebooks/neural-edge-video-compression")
if preferred_drive_repo.exists() and REPO_DIR != preferred_drive_repo:
    print(f"Switching to Drive repo at {preferred_drive_repo}")
    REPO_DIR = preferred_drive_repo
    if not Path(VIDEO_PATH).exists():
        VIDEO_PATH = str(REPO_DIR / "video.mp4")

if REPO_DIR.exists():
    print(f"Reusing existing repo at {REPO_DIR}")
else:
    if not REPO_URL:
        raise RuntimeError("REPO_DIR is missing and REPO_URL is empty. Set one of them before continuing.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)
print({"repo_dir": str(REPO_DIR), "video_path": VIDEO_PATH})


In [ ]:
import subprocess
import sys

requirements_path = REPO_DIR / "research/roi_budgeting/requirements.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path), "ultralytics"],
    check=True,
)
subprocess.run(["nvidia-smi"], check=False)

import torch
print({
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_device_count": int(torch.cuda.device_count()),
})


In [ ]:
import json
from pathlib import Path

import yaml

video_path = Path(VIDEO_PATH).expanduser().resolve()
if not video_path.exists():
    fallback_video_path = (REPO_DIR / "video.mp4").resolve()
    if fallback_video_path.exists():
        print(f"VIDEO_PATH not found, falling back to {fallback_video_path}")
        video_path = fallback_video_path
        VIDEO_PATH = str(video_path)
    else:
        raise FileNotFoundError(f"Set VIDEO_PATH to an existing clip before continuing: {video_path}")

amt_weights_path = REPO_DIR / "models/amt-s.pth"
amt_repo_dir = REPO_DIR / "_third_party_amt"
if not amt_weights_path.exists():
    raise FileNotFoundError(f"AMT weights not found: {amt_weights_path}")
if not amt_repo_dir.exists():
    raise FileNotFoundError(f"AMT repo not found: {amt_repo_dir}")

research_dir = REPO_DIR / "research/roi_budgeting"
results_dir = research_dir / "results"
baseline_dir = results_dir / "baseline/frame_removal"
manifests_dir = results_dir / "manifests"
baseline_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)

base_compression_cfg = yaml.safe_load((REPO_DIR / "configs/gpu/compression.yaml").read_text())
base_compression_cfg["roi_detection"]["runtime"]["device"] = "cuda:0"
base_compression_cfg["roi_detection"]["runtime"]["half"] = True
base_compression_cfg["compression"]["dcvc"]["device"] = "cuda"
base_compression_cfg["compression"]["dcvc"]["use_cuda"] = True
base_compression_cfg["output"]["out_dir"] = "research/roi_budgeting/results/baseline/compression"

compression_runtime_cfg = research_dir / "configs/colab_runtime_compression.yaml"
compression_runtime_cfg.write_text(yaml.safe_dump(base_compression_cfg, sort_keys=False), encoding="utf-8")

research_cfg = yaml.safe_load((research_dir / "configs/colab.yaml").read_text())
research_cfg["paths"]["repo_root"] = str(REPO_DIR)
research_cfg["paths"]["output_dir"] = str(results_dir)
research_cfg["paths"]["video_path"] = str(video_path)
research_cfg["paths"]["roi_detections_json"] = str(baseline_dir / "roi_detections.json")
research_cfg["paths"]["frame_drop_json"] = str(baseline_dir / "frame_drop.json")
research_cfg["paths"]["amt_probe_manifest"] = str(manifests_dir / "amt_probe_manifest.json")
research_cfg["signals"]["amt_risk"]["weights_path"] = str(amt_weights_path)
research_cfg["signals"]["amt_risk"]["repo_dir"] = str(amt_repo_dir)
research_cfg["budget"]["roi_target_kbps"] = float(ROI_TARGET_KBPS)

research_runtime_cfg = research_dir / "configs/colab_runtime.yaml"
research_runtime_cfg.write_text(yaml.safe_dump(research_cfg, sort_keys=False), encoding="utf-8")

print(json.dumps({
    "compression_runtime_cfg": str(compression_runtime_cfg),
    "research_runtime_cfg": str(research_runtime_cfg),
    "video_path": str(video_path),
    "amt_weights_path": str(amt_weights_path),
    "amt_repo_dir": str(amt_repo_dir),
    "results_dir": str(results_dir),
}, indent=2))


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "scripts/test_frame_removal.py",
        "--config",
        "research/roi_budgeting/configs/colab_runtime_compression.yaml",
        "--video",
        VIDEO_PATH,
        "--out-dir",
        "research/roi_budgeting/results/baseline/frame_removal",
        "--no-viz",
    ],
    cwd=REPO_DIR,
    check=True,
)


In [ ]:
import subprocess
import sys

research_dir = REPO_DIR / "research/roi_budgeting"

def run_offline(experiment_config=None):
    cmd = [
        sys.executable,
        "-m",
        "roi_budgeting.runners.run_offline_eval",
        "--config",
        "configs/colab_runtime.yaml",
    ]
    if experiment_config:
        cmd.extend(["--experiment-config", experiment_config])
    subprocess.run(cmd, cwd=research_dir, check=True)

run_offline()
if RUN_V1:
    run_offline("configs/experiments/v1_motion_only.yaml")
if RUN_V2:
    run_offline("configs/experiments/v2_motion_uncertainty.yaml")


In [ ]:
import subprocess
import sys

research_dir = REPO_DIR / "research/roi_budgeting"

subprocess.run(
    [
        sys.executable,
        "-m",
        "roi_budgeting.runners.run_amt_probes",
        "--config",
        "configs/colab_runtime.yaml",
    ],
    cwd=research_dir,
    check=True,
)

if RUN_V3:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "roi_budgeting.runners.run_offline_eval",
            "--config",
            "configs/colab_runtime.yaml",
            "--experiment-config",
            "configs/experiments/v3_motion_uncertainty_amt.yaml",
        ],
        cwd=research_dir,
        check=True,
    )


In [ ]:
import json
from pathlib import Path

import pandas as pd

manifests_dir = REPO_DIR / "research/roi_budgeting/results/manifests"
manifest_paths = [
    manifests_dir / "offline_eval_summary.json",
    manifests_dir / "v1_motion_only_summary.json",
    manifests_dir / "v2_motion_uncertainty_summary.json",
    manifests_dir / "v3_motion_uncertainty_amt_summary.json",
]

rows = []
for path in manifest_paths:
    if not path.exists():
        continue
    payload = json.loads(path.read_text())
    row = {
        "manifest": path.name,
        "status": payload.get("status", "ok"),
        "baseline_roi_kept": ((payload.get("frame_drop", {}) or {}).get("roi_kept_frames")),
    }
    experiment = payload.get("experiment") or {}
    comparison = experiment.get("comparison") or {}
    proposal = experiment.get("proposal") or {}
    row.update({
        "experiment_name": experiment.get("name"),
        "proposed_roi_kept": proposal.get("roi_kept_frames"),
        "jaccard_vs_baseline": ((comparison.get("overlap") or {}).get("jaccard")),
        "motion_delta": comparison.get("mean_motion_score_delta"),
        "uncertainty_delta": comparison.get("mean_uncertainty_score_delta"),
        "amt_delta": comparison.get("mean_amt_risk_delta"),
        "combined_delta": comparison.get("mean_combined_score_delta"),
        "amt_source": ((experiment.get("amt_probe") or {}).get("source")),
    })
    rows.append(row)

df = pd.DataFrame(rows)
df


In [ ]:
import shutil
from pathlib import Path

results_dir = REPO_DIR / "research/roi_budgeting/results"
destination = Path(DRIVE_RESULTS_DIR).expanduser()

def copy_tree(src: Path, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    for path in src.rglob("*"):
        rel = path.relative_to(src)
        target = dst / rel
        if path.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

if DRIVE_RESULTS_DIR:
    copy_tree(results_dir, destination)
    print(f"Synced results to {destination}")
else:
    print("Skipping Drive sync because DRIVE_RESULTS_DIR is empty.")
